# Stage 1: Simulation-Based Pretraining (Shuffled Timestamps)

Following ViPer-RV (Gavankar et al. 2025, AJ) two-stage training methodology:
- Stage 1: Shuffled timestamps -> model learns Keplerian Doppler shapes
- Stage 2 (pretrain_stage2.ipynb): Ordered timestamps -> model adapts to real cadence

50,000 simulated RV stars with real HARPS/HIRES cadences, GP quasi-periodic stellar
activity (ExoplANNET methodology), and injected Keplerian planet signals (50% positive).

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "0"

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import math
import random
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR
from sklearn.metrics import roc_auc_score

seed = 42
random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

observations = pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl')
print(f"Real obs: {len(observations)}, stars: {observations['star_name'].nunique()}")

from sim_dataset import get_sim_loaders
print("Generating 50,000 simulated stars (Stage 1: shuffled)...")
train_loader, val_loader, feat_mean, feat_std = get_sim_loaders(
    observations, n_sim=50000, batch_size=32, mode="shuffled", seed=seed, device=str(device))
np.save('/kaggle/working/sim_norm_mean.npy', feat_mean)
np.save('/kaggle/working/sim_norm_std.npy', feat_std)

from transformer_model import ExoplanetTransformer
model = ExoplanetTransformer(feat_dim=21, d_model=48, nhead=4, num_layers=1,
                              dim_feedforward=96, dropout=0.3).to(device)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Training loop — Stage 1: Shuffled simulated data
# Balanced 50/50 simulated data, no pos_weight needed
criterion = nn.BCEWithLogitsLoss()

optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=5e-3)
warmup_epochs = 5
total_epochs = 30

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)
train_losses, val_losses = [], []
train_aucs, val_aucs = [], []
best_val_auc = 0
best_model_state = None

for epoch in range(total_epochs):
    model.train()
    train_loss = 0
    all_train_probs, all_train_labels = [], []
    for padded, mask, labels in train_loader:
        padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(padded, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * padded.size(0)
        all_train_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())
    scheduler.step()
    train_auc = roc_auc_score(all_train_labels, all_train_probs)

    model.eval()
    val_loss = 0
    all_val_probs, all_val_labels = [], []
    with torch.no_grad():
        for padded, mask, labels in val_loader:
            padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
            logits = model(padded, mask)
            val_loss += criterion(logits, labels).item() * padded.size(0)
            all_val_probs.extend(torch.sigmoid(logits).cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())
    val_auc = roc_auc_score(all_val_labels, all_val_probs)

    train_losses.append(train_loss / len(train_loader.dataset))
    val_losses.append(val_loss / len(val_loader.dataset))
    train_aucs.append(train_auc); val_aucs.append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch+1}/{total_epochs} | Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}')

print(f'\nBest val AUC: {best_val_auc:.4f}')
torch.save(best_model_state, '/kaggle/working/pretrained_stage1.pth')
print('Saved pretrained Stage 1 weights.')

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.set_title('Stage 1: Shuffled Simulated — Loss')
ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(train_aucs, label='Train AUC')
ax2.plot(val_aucs, label='Val AUC')
ax2.set_title('Stage 1: Shuffled Simulated — ROC-AUC')
ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()